# Phase 3 - Feature engineering
**Objective:** turn the joined original tables into model inputs with a *reusable, training-fitted* preprocessing pipeline, and document what is used, what is excluded and why. No final classifier is trained here; the only model is a small LightGBM used in section 9 to test, honestly, whether the engineered features help.

| Plan step (Phase 3 plan for this ZIP) | Section |
|---|---|
| 1 Supplied return-request columns | 2, 3 |
| 2 Join `products.csv` (unique key, coverage) | 2 |
| 3 Join `customers.csv`, check tenure, evaluate `city_tier` | 2, 5 |
| 4 Meaning of each missing value | 4 |
| 5 Size-chart measurements | 4 |
| Leakage protection | 6 |
| Training-fitted matrices for train / calibration / test | 7 |
| Feature-quality report | 8 |
| Does it help? | 9 |

**How to run:** keep this file in `FULL_CODE/notebooks/` with `src/r2sorter/preprocess.py` and the five official CSVs in `FULL_CODE/data/`.

In [1]:
import sys
from pathlib import Path

for candidate in (Path("../src"), Path("src")):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from r2sorter.config import load_config
from r2sorter.data import load_all
from r2sorter.features import build_feature_table, tabular_columns
from r2sorter.preprocess import (
    CATEGORICAL_FEATURES,
    FeaturePreprocessor,
    add_engineered_features,
    data_quality_flags,
    excluded_columns,
    feature_catalogue,
    feature_quality_report,
    validate_joins,
)
from r2sorter.splits import chronological_split, describe_splits

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 120)
cfg = load_config()
data = load_all(cfg)
train, test = data["train"], data["test"]
products, charts, customers = data["products"], data["size_charts"], data["customers"]
REPORTS = Path("../reports") if Path("../reports").exists() else Path("reports")
REPORTS.mkdir(exist_ok=True)
print(f"train {len(train):,}  test {len(test):,}  products {len(products):,}  customers {len(customers):,}")

train 43,068  test 16,932  products 975  customers 20,205


## 2. Join validation (before any merge)
Keys must be unique and every request must find its product and customer. A failed check stops the pipeline.

In [2]:
joins = validate_joins(pd.concat([train, test]), products, customers, charts)
joins.to_csv(REPORTS / "phase3_join_validation.csv", header=["value"])
assert (
    joins[
        [
            "product_id_duplicates",
            "customer_id_duplicates",
            "requests_without_product",
            "requests_without_customer",
            "products_with_unknown_chart",
        ]
    ].sum()
    == 0
)
joins.to_frame("value")

,value
product_id_duplicates,0
customer_id_duplicates,0
requests_without_product,0
requests_without_customer,0
products_with_unknown_chart,0
products_without_chart,90


`products_without_chart` is expected: free-size products (sarees, dupattas) have no size chart. It is **chart unavailable**, not suspicious behaviour, so it becomes the neutral flag `size_chart_missing` and never a fraud signal.

## 3. Feature catalogue and excluded columns
Every engineered feature lists its source, rule and why it cannot leak.

In [3]:
catalogue = feature_catalogue()
catalogue.to_csv(REPORTS / "phase3_feature_catalogue.csv")
print(len(catalogue), "engineered / derived features")
catalogue

65 engineered / derived features


,group,source,rule,leakage_safety
feature,,,,
gender,product,products.gender,one-hot product gender,static catalogue fact
delivery_days,original,delivery_days,as given,row's own columns only
days_since_delivery,original,days_since_delivery,as given,row's own columns only
order_value_inr,original,order_value_inr,as given,row's own columns only
item_price_inr,original,item_price_inr,as given,row's own columns only
discount_pct,original,discount_pct,as given,row's own columns only
items_in_order,original,items_in_order,as given,row's own columns only
customer_tenure_days,original,customer_tenure_days,as given (also a fairness group),row's own columns only
prior_orders_count,original,prior_orders_count,as given,given by the retailer at request time


In [4]:
excluded = excluded_columns()
excluded.to_csv(REPORTS / "phase3_excluded_columns.csv")
excluded

,reason
column,
return_id,identifier - carries no behaviour and would le...
order_id,identifier
customer_id,identifier (customer history is expressed thro...
product_id,identifier (replaced by the training-fitted pr...
order_date,raw date - converted to days_since_order and r...
delivery_date,raw date - converted to timing features
return_request_date,raw date and the split clock; month is not use...
label,TARGET - never an input
abusive,TARGET-derived


## 4. What a missing value means

In [5]:
# prior_return_rate: expected missing exactly when there is no history
missing_rate = train["prior_return_rate"].isna()
no_history = train["prior_orders_count"] == 0
print("prior_return_rate missing iff prior_orders_count == 0:", bool((missing_rate == no_history).all()))
print(
    f"missing share {missing_rate.mean():.3%}; abuse rate when history missing {(train.label[missing_rate] == 'abusive').mean():.3%} "
    f"vs {(train.label[~missing_rate] == 'abusive').mean():.3%} otherwise"
)
# size charts: which measurements apply to which product kind
kinds = products.merge(charts.drop_duplicates("size_chart_id"), on="size_chart_id", how="left")
measures = ["chest_cm", "waist_cm", "hip_cm", "shoulder_cm", "length_cm", "inseam_cm", "foot_length_cm"]
print("\nshare of charts with the measure present, by product category:")
chart_rows = charts.merge(products.drop_duplicates("size_chart_id")[["size_chart_id", "category"]], on="size_chart_id")
chart_rows.groupby("category")[measures].agg(lambda s: s.notna().mean()).round(2)

prior_return_rate missing iff prior_orders_count == 0: True
missing share 8.723%; abuse rate when history missing 7.533% vs 7.830% otherwise

share of charts with the measure present, by product category:


,chest_cm,waist_cm,hip_cm,shoulder_cm,length_cm,inseam_cm,foot_length_cm
category,,,,,,,
Footwear,0.00,0.00,0.00,0.00,0.00,0.00,1.0
Men Ethnic,1.00,0.00,0.00,1.00,1.00,0.00,0.0
Men Western,0.55,0.45,0.45,0.55,0.55,0.45,0.0
Women Ethnic,0.86,1.00,1.00,0.76,1.00,0.00,0.0
Women Western,0.65,1.00,1.00,0.65,0.80,0.20,0.0


Measurements are **not** imputed: a missing `inseam_cm` on a top or `foot_length_cm` on clothing means *not applicable*. `size_chart_deviation` uses the first measurement that exists for the chart (chest, waist, hip, shoulder, length, foot) and is NaN otherwise. It compares the ordered size with the chart's *median* size - a property of the **garment chart**, not of the customer's body.

## 5. Customer table: tenure consistency and the `city_tier` question

In [6]:
frame = build_feature_table(train, test, products, cfg)
frame = add_engineered_features(frame, charts, cfg, customers)
print("tenure recomputed from signup_date minus supplied customer_tenure_days (days):")
print(frame["tenure_gap_days"].describe().round(3).to_string())
labelled = frame[frame["is_train"]].assign(abusive=lambda d: (d["label"] == "abusive").astype(int))
print("\nabuse rate by city_tier:")
labelled.groupby("city_tier")["abusive"].agg(["mean", "size"]).round(4)

tenure recomputed from signup_date minus supplied customer_tenure_days (days):
count    60000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0

abuse rate by city_tier:


,mean,size
city_tier,,
1,0.0793,18503
2,0.0721,14528
3,0.0844,10037


The supplied `customer_tenure_days` agrees with `signup_date` on every row, so we keep the supplied column and do not add `signup_date` as a second copy. `city_tier` shows only a weak abuse gap and is a possible proxy for customer group, so it is **kept out of the default features** and tested as a separate arm in section 9.

## 6. Leakage tests

In [7]:
# 6a. the label never reaches the matrix
pre_probe = FeaturePreprocessor(cfg).fit(frame[frame["is_train"]].head(5000))
cols = set(pre_probe.transform_tree(frame.head(10)).columns)
forbidden = {"label", "abusive", "customer_reason", "return_id", "order_id", "customer_id", "product_id"}
assert not (cols & forbidden) and not [c for c in cols if c.startswith("lab_")], cols & forbidden
print("6a ok - no label, identifier, raw text or lab_* column among", len(cols), "model columns")

# 6b. truncation: features of earlier rows must not change when later rows are removed
cut = frame["return_request_date"].quantile(0.5)
early = frame[frame["return_request_date"] <= cut]
short = add_engineered_features(
    build_feature_table(train[train["return_request_date"] <= cut], test.iloc[:0], products, cfg),
    charts,
    cfg,
    customers,
)
check = ["refund_value_30d", "refund_value_60d", "refund_value_90d", "prior_return_count_est", "hist_n_prev"]
key = ["return_id"]
merged = early[key + check].merge(short[key + check], on=key, suffixes=("_full", "_cut"))
for name in check:
    assert np.allclose(merged[f"{name}_full"], merged[f"{name}_cut"], equal_nan=True), name
print(f"6b ok - {len(merged):,} early rows identical with and without the future")

# 6c. the preprocessor learns only from the rows it is fitted on
fit_rows = frame[frame["is_train"]].head(20000)
pre = FeaturePreprocessor(cfg).fit(fit_rows)
before = pre.learned_parameters()
_ = pre.transform_linear(frame[~frame["is_train"]])
assert pre.learned_parameters() == before
print("6c ok - transforming test rows changed nothing that was learned")

6a ok - no label, identifier, raw text or lab_* column among 56 model columns


6b ok - 30,183 early rows identical with and without the future


6c ok - transforming test rows changed nothing that was learned


Same-day ordering caveat from Phase 2 still applies: requests on one day are ordered by `return_id`, so 'earlier' is exact only between days.

## 7. Training-fitted matrices for the plan split and the final test

In [8]:
labelled = frame[frame["is_train"]].reset_index(drop=True)
plan = chronological_split(labelled["return_request_date"])
display(describe_splits(labelled, plan))
pre = FeaturePreprocessor(cfg).fit(labelled[plan == "train"])
parts = {
    "train": labelled[plan == "train"],
    "calibration": labelled[plan == "calibration"],
    "validation_test": labelled[plan == "test"],
    "final_test_Jul-Sep": frame[~frame["is_train"]],
}
summary = []
for name, rows in parts.items():
    linear, tree = pre.transform_linear(rows), pre.transform_tree(rows)
    assert linear.isna().sum().sum() == 0 and np.isfinite(linear.to_numpy()).all()
    assert list(linear.columns) == list(pre.transform_linear(parts["train"].head(3)).columns)
    summary.append(
        {
            "part": name,
            "rows": len(rows),
            "linear_cols": linear.shape[1],
            "tree_cols": tree.shape[1],
            "rows_with_unseen_category": int((linear["n_unseen_categories"] > 0).sum()),
        }
    )
print("dropped as constant or duplicate on the training matrix:", pre.dropped_columns_)
pd.DataFrame(summary).set_index("part")

,first_date,last_date,rows,share_of_rows,days,abusive,genuine_quality_issue,genuine_fit_issue,genuine_wrong_item
split,,,,,,,,,
train,2025-10-01,2026-04-07,30183,0.701,189,0.079,0.275,0.477,0.169
calibration,2026-04-08,2026-05-07,4388,0.102,30,0.065,0.294,0.488,0.152
test,2026-05-08,2026-06-30,8497,0.197,54,0.082,0.289,0.493,0.136


dropped as constant or duplicate on the training matrix: {'missingindicator_hist_mean_item_price': 'exact duplicate of missingindicator_hist_days_since_prev', 'missingindicator_hist_price_ratio': 'exact duplicate of missingindicator_hist_days_since_prev', 'catalogue_size_match': 'constant in the training rows', 'invalid_date_order': 'constant in the training rows', 'price_exceeds_mrp': 'constant in the training rows', 'missing_item_price': 'constant in the training rows', 'size_kind_free': 'exact duplicate of missingindicator_size_chart_deviation', 'size_kind_shoe': 'exact duplicate of category_Footwear', 'fit_type_free size': 'exact duplicate of missingindicator_size_chart_deviation'}


,rows,linear_cols,tree_cols,rows_with_unseen_category
part,,,,
train,30183,175,61,0
calibration,4388,175,61,0
validation_test,8497,175,61,0
final_test_Jul-Sep,16932,175,61,0


Columns dropped by the redundancy step are guards that are constant in this data (e.g. every ordered size is sold, no invalid dates). They stay in the code so a future bad record is caught by `data_quality_flags`, but they carry no information today.

In [9]:
data_quality_flags(parts["final_test_Jul-Sep"])

invalid_date_order            0
missing_item_price            0
price_exceeds_mrp             0
size_not_in_catalogue         0
outside_return_window         0
tenure_mismatch_over_1_day    0
dtype: int64

## 8. Feature-quality report

In [10]:
report = feature_quality_report(pre.transform_tree(parts["train"]))
report.to_csv(REPORTS / "phase3_feature_quality_report.csv")
print("constant:", report.index[report["constant"]].tolist())
print("duplicate of earlier column:", report.index[report["duplicate_of_earlier_column"]].tolist())
print("any inf:", report.index[report["has_inf"]].tolist())
print("\ncategory coverage of the final test set (share of rows whose category was never seen in training):")
unseen = {
    c: float((~parts["final_test_Jul-Sep"][c].astype(str).isin(pre._categories[c])).mean())
    for c in CATEGORICAL_FEATURES
}
print(pd.Series(unseen).round(4).to_string())
report.sort_values("missing_%", ascending=False).head(15)

constant: ['n_unseen_categories']
duplicate of earlier column: []
any inf: []

category coverage of the final test set (share of rows whose category was never seen in training):
category             0.0
sub_category         0.0
payment_method       0.0
fulfilment_centre    0.0
size_kind            0.0
brand                0.0
gender               0.0
fabric               0.0
fit_type             0.0
request_dow          0.0


,dtype,missing_%,distinct,constant,duplicate_of_earlier_column,has_inf
hist_days_since_prev,float64,46.804,184,False,False,False
hist_mean_item_price,float64,46.804,4825,False,False,False
hist_price_ratio,float64,46.804,14581,False,False,False
prior_return_rate,float64,9.244,646,False,False,False
size_chart_deviation,float64,7.103,385,False,False,False
discount_pct,int64,0.000,16,False,False,False
items_in_order,int64,0.000,5,False,False,False
customer_tenure_days,int64,0.000,2376,False,False,False
prior_orders_count,int64,0.000,131,False,False,False
mrp_inr,int64,0.000,84,False,False,False


## 9. Does it help? (honest ablation)
Structured features only (no text model), tabular LightGBM, trained on requests before 1 May 2026 and scored on May-June 2026, 3 seeds, preprocessing fitted on the training rows only. **A = the existing structured columns**, B = A plus the engineered columns, C = B plus `city_tier`.

Caveat: the May-June window was also used in earlier tuning, so absolute numbers may be optimistic; the comparison between arms is what matters.

In [11]:
split_date = pd.Timestamp(cfg["validation"]["holdout_start"])
fit_rows = labelled[labelled["return_request_date"] < split_date]
eval_rows = labelled[labelled["return_request_date"] >= split_date]
y_fit, y_eval = (fit_rows["label"] == "abusive").astype(int), (eval_rows["label"] == "abusive").astype(int)

base_cols = [c for c in tabular_columns(labelled) if not c.startswith(("lab_", "ev_", "txt_"))]
pre_ab = FeaturePreprocessor(cfg).fit(fit_rows)
tree_fit, tree_eval = pre_ab.transform_tree(fit_rows), pre_ab.transform_tree(eval_rows)
extra = [c for c in tree_fit.columns if c not in base_cols]


def arm(rows, tree, with_extra, with_tier):
    out = rows[base_cols].copy()
    if with_extra:
        out = pd.concat([out, tree[extra]], axis=1)
    if with_tier:
        out["city_tier"] = rows["city_tier"].to_numpy()
    return out


arms = {
    "A baseline structured": (False, False),
    "B + engineered": (True, False),
    "C + engineered + city_tier": (True, True),
}
results = []
for label, (with_extra, with_tier) in arms.items():
    xf, xe = arm(fit_rows, tree_fit, with_extra, with_tier), arm(eval_rows, tree_eval, with_extra, with_tier)
    for seed in (0, 1, 2):
        model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=30,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=8,
            random_state=seed,
            verbose=-1,
        )
        model.fit(xf, y_fit)
        p = model.predict_proba(xe)[:, 1]
        results.append(
            {
                "arm": label,
                "seed": seed,
                "features": xf.shape[1],
                "AP": average_precision_score(y_eval, p),
                "log_loss": log_loss(y_eval, p),
                "brier": brier_score_loss(y_eval, p),
            }
        )
results = pd.DataFrame(results)
ablation = (
    results.groupby("arm")
    .agg(
        features=("features", "first"),
        AP=("AP", "mean"),
        AP_sd=("AP", "std"),
        log_loss=("log_loss", "mean"),
        brier=("brier", "mean"),
    )
    .round(4)
)
ablation.to_csv(REPORTS / "phase3_ablation.csv")
ablation

,features,AP,AP_sd,log_loss,brier
arm,,,,,
A baseline structured,37,0.6920,0.0045,0.1502,0.0412
B + engineered,64,0.6899,0.0018,0.1507,0.0415
C + engineered + city_tier,65,0.6911,0.0017,0.1501,0.0413


In [12]:
gain = ablation.loc["B + engineered", "AP"] - ablation.loc["A baseline structured", "AP"]
noise = max(ablation["AP_sd"].max(), 1e-9)
print(f"AP change from engineered features: {gain:+.4f} (seed sd up to {noise:.4f})")
print("Treat as a real gain only if the change is clearly larger than the seed spread and log-loss also improves.")

AP change from engineered features: -0.0021 (seed sd up to 0.0045)
Treat as a real gain only if the change is clearly larger than the seed spread and log-loss also improves.


## 10. Verdict

In [13]:
base, eng, tier = (ablation.loc[k] for k in ablation.index.sort_values())
margin = 2 * noise
gain_eng, gain_tier = eng.AP - base.AP, tier.AP - eng.AP
helps = gain_eng > margin and eng.log_loss < base.log_loss
print(
    "Engineered features: "
    + ("IMPROVE" if helps else "do NOT clearly improve")
    + f" the structured baseline (dAP {gain_eng:+.4f}, margin {margin:.4f})."
)
print(
    "city_tier: "
    + ("adds value" if gain_tier > margin else "adds no clear value")
    + f" (dAP {gain_tier:+.4f}); it stays out of the default set because of proxy risk."
)

Engineered features: do NOT clearly improve the structured baseline (dAP -0.0021, margin 0.0090).
city_tier: adds no clear value (dAP +0.0012); it stays out of the default set because of proxy risk.
